In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 01 - Data Loading
## Step 1: Inspect a single file
Before merging all 20 files, we inspect one file to understand its structure.

In [5]:
df = pd.read_csv("../data/raw/amsterdam_weekdays.csv")
print(df.shape)

(1103, 20)


In [6]:
df.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
0,0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,1,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772
1,1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,1,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432
2,2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,1,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103
3,3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,2,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663
4,4,485.552926,Private room,False,True,2.0,True,0,0,10.0,98.0,1,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1103 entries, 0 to 1102
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  1103 non-null   int64  
 1   realSum                     1103 non-null   float64
 2   room_type                   1103 non-null   object 
 3   room_shared                 1103 non-null   bool   
 4   room_private                1103 non-null   bool   
 5   person_capacity             1103 non-null   float64
 6   host_is_superhost           1103 non-null   bool   
 7   multi                       1103 non-null   int64  
 8   biz                         1103 non-null   int64  
 9   cleanliness_rating          1103 non-null   float64
 10  guest_satisfaction_overall  1103 non-null   float64
 11  bedrooms                    1103 non-null   int64  
 12  dist                        1103 non-null   float64
 13  metro_dist                  1103 

## Step 2: List all raw files

In [8]:
from pathlib import Path

RAW_DIR = Path("../data/raw")
files = sorted(RAW_DIR.glob("*.csv"))

print("Number of files:", len(files))
for f in files:
    print(f.name)

Number of files: 20
amsterdam_weekdays.csv
amsterdam_weekends.csv
athens_weekdays.csv
athens_weekends.csv
barcelona_weekdays.csv
barcelona_weekends.csv
berlin_weekdays.csv
berlin_weekends.csv
budapest_weekdays.csv
budapest_weekends.csv
lisbon_weekdays.csv
lisbon_weekends.csv
london_weekdays.csv
london_weekends.csv
paris_weekdays.csv
paris_weekends.csv
rome_weekdays.csv
rome_weekends.csv
vienna_weekdays.csv
vienna_weekends.csv


## Step 3: Check that all files share the same columns

In [9]:
reference_cols = list(pd.read_csv(files[0], nrows=0).columns)

all_match = True
for f in files:
    cols = list(pd.read_csv(f, nrows=0).columns)
    if cols != reference_cols:
        all_match = False
        print("Different columns in:", f.name, set(cols) ^ set(reference_cols))

print("All files have identical columns:", all_match)

All files have identical columns: True


In [10]:
sample = files[0]
print(sample.name)              # amsterdam_weekdays.csv
print(sample.stem)              # amsterdam_weekdays   (الاسم من غير الامتداد)
print(sample.stem.split("_"))   # ['amsterdam', 'weekdays']

amsterdam_weekdays.csv
amsterdam_weekdays
['amsterdam', 'weekdays']


## Step 4: Load all files and combine them into one master dataset
City and day type exist only in the file names, so we add them as columns before combining.

In [11]:
frames = []

for f in files:
    city, day_type = f.stem.split("_")
    temp = pd.read_csv(f)
    temp["city"] = city.title()     # amsterdam -> Amsterdam
    temp["day_type"] = day_type     # weekdays / weekends
    frames.append(temp)

df_all = pd.concat(frames, ignore_index=True)
print(df_all.shape)

(51707, 22)


In [13]:
total_rows = sum(len(pd.read_csv(f)) for f in files)
print("Sum of rows in all files:", total_rows)
print("Rows in combined dataset:", len(df_all))
print("Match:", total_rows == len(df_all))

Sum of rows in all files: 51707
Rows in combined dataset: 51707
Match: True


In [14]:
pd.crosstab(df_all["city"], df_all["day_type"], margins=True)

day_type,weekdays,weekends,All
city,,,
Amsterdam,1103,977,2080
Athens,2653,2627,5280
Barcelona,1555,1278,2833
Berlin,1284,1200,2484
Budapest,2074,1948,4022
Lisbon,2857,2906,5763
London,4614,5379,9993
Paris,3130,3558,6688
Rome,4492,4535,9027


In [15]:
df_all.to_csv("../data/processed/airbnb_master_raw.csv", index=False)
print("Saved:", df_all.shape)

Saved: (51707, 22)
